# TR Causal Variant Analysis — AoU Researcher Workbench

End-to-end pipeline for fine-mapping (CAVIAR) and conditional analysis (REGENIE) of tandem-repeat / trait pairs in *All of Us*.

**Pipeline phases**
1. Filter TR–trait pairs to those with upstream REGENIE output (`00`)
2. Per-chromosome SNP filter from ACAF PGEN (`01`)
3. Genome-wide merge + LD prune for REGENIE step 1 (`02`)
4. Build per-threshold TR PGEN across EUR/AFR/AMR (`03`)
5. Per (TR, PHENO) ±250 kb extraction + drop-SNP list (`04`)
6. REGENIE step 1 — genome-wide LD-pruned model (`05`)
7. REGENIE step 2 — locus association (`06`)

**Fine mapping (CAVIAR)**
  8. Z-score table for top-100 SNPs + TR (`07`)
  9. LD matrix per pair (`08`)
 10. CAVIAR (`09`)

**Conditional analysis**
 11. Extract lead SNP per pair → genome-wide exclude list (`10`)
 12. REGENIE step 1 excluding lead SNPs (`11`)
 13. REGENIE step 2 conditioning on lead SNP (`12`)
 14. REGENIE step 2 conditioning on TR (`13`)

All paths, binaries, and covariate strings are centralised in `config.sh`. Shared bash functions live in `common.sh`. Every `%%bash` cell sources these two files; nothing else carries state between cells.

**Before running:** edit `config.sh` and set `COVAR_FILE` (or `export AOU_COVAR_FILE=...` in your shell) to point at your workspace's covariate TSV. The default value contains a placeholder.

---
## Sanity check — config loads cleanly
Run this first; if it errors, fix `config.sh` before continuing.

In [ ]:
%%bash
source ./config.sh
source ./common.sh
echo "PROJECT_ROOT = $PROJECT_ROOT"
echo "REGENIE_BIN  = $REGENIE_BIN"
echo "COVAR_FILE   = $COVAR_FILE"
command -v plink2 >/dev/null && echo "plink2 OK" || echo "plink2 MISSING"
command -v plink  >/dev/null && echo "plink  OK" || echo "plink  MISSING"
[[ -x "$REGENIE_BIN" ]] && echo "regenie OK"  || echo "regenie MISSING at $REGENIE_BIN"
[[ -x "$CAVIAR_BIN"  ]] && echo "caviar  OK"  || echo "caviar  MISSING at $CAVIAR_BIN"

---
## 00 — Filter TR–trait pairs to those with upstream REGENIE output, generate per-locus PhenoList and per-threshold TRList files
Reads `$TR_TRAIT_PAIRS_RAW`, drops any pair whose lab-wide REGENIE file is missing or whose TR row isn't present in that file, and writes the filtered list + a missing-pairs log.

Reads `$TR_TRAIT_PAIRS` and produces two sets of lookup files:
- `{PROJECT_ROOT}/{Locus}_Binary_PhenoList` — one file per TR locus listing its distinct phenotypes (no header). Consumed by steps 04 and 06.
- `{PROJECT_ROOT}/TRList_{cutoff}` — one file per cutoff threshold listing its distinct loci (no header). Consumed by step 03.

In [ ]:
%%bash
source ./config.sh
source ./common.sh

: > "$MISSING_LIST"
head -n 1 "$TR_TRAIT_PAIRS_RAW" > "$TR_TRAIT_PAIRS"

tail -n +2 "$TR_TRAIT_PAIRS_RAW" | while IFS= read -r line; do
    IFS=$'\t' read -r TR THRESH PHENO _ <<< "$line"
    TRFILE="${UPSTREAM_REGENIE_DIR}/AoU.AllAncestries.${THRESH}.${PHENO}.MACounts2.CombinedCohorts.regenie.edited.txt"

    if [[ ! -f "$TRFILE" ]]; then
        printf '%s\t%s\t%s\n' "$TR" "$PHENO" 'File completely missing' >> "$MISSING_LIST"
        continue
    fi

    if grep -qw "$TR" "$TRFILE"; then
        printf '%s\n' "$line" >> "$TR_TRAIT_PAIRS"
    else
        printf '%s\t%s\t%s\n' "$TR" "$PHENO" 'No output in file' >> "$MISSING_LIST"
    fi
done

echo "Pairs kept:    $(($(wc -l < "$TR_TRAIT_PAIRS") - 1))"
echo "Pairs missing: $(wc -l < "$MISSING_LIST")"

# per-locus PhenoList files
unique_trs | while IFS= read -r TR; do
    : > "${PROJECT_ROOT}/${TR}_Binary_PhenoList"
done

awk -F'\t' 'NR>1 {print $1"\t"$3}' "$TR_TRAIT_PAIRS" \
    | sort -u \
    | while IFS=$'\t' read -r TR PHENO; do
        printf '%s\n' "$PHENO" >> "${PROJECT_ROOT}/${TR}_Binary_PhenoList"
    done

# per-threshold TRList files
awk -F'\t' 'NR>1 {print $2}' "$TR_TRAIT_PAIRS" | sort -u \
    | while IFS= read -r CUTOFF; do
        : > "${PROJECT_ROOT}/TRList_${CUTOFF}"
    done

awk -F'\t' 'NR>1 {print $2"\t"$1}' "$TR_TRAIT_PAIRS" \
    | sort -u \
    | while IFS=$'\t' read -r CUTOFF LOCUS; do
        printf '%s\n' "$LOCUS" >> "${PROJECT_ROOT}/TRList_${CUTOFF}"
    done

echo "PhenoList files:"
ls "${PROJECT_ROOT}/"*_Binary_PhenoList 2>/dev/null | wc -l
echo "TRList files:"
ls "${PROJECT_ROOT}/"TRList_* 2>/dev/null | wc -l

---
## 01 — Per-chromosome SNP filter from ACAF PGEN
Extract variants in TR ±250 kb regions, restrict to the 88k sample list, apply MAC/missingness/HWE filters. Chromosomes are derived from the BED (no hard-coded list).

In [ ]:
%%bash
source ./config.sh
source ./common.sh

for CHR in $(chroms_from_bed "$REGIONS_BED"); do
    echo "=== chr${CHR} ==="
    PGEN="${SNP_DIR}/acaf_threshold.chr${CHR}"
    OUT="${SNP_DIR}/AoU.chr${CHR}.filter"
    
    if [ -f "${PGEN}.pgen" ]; then
        plink2 \
            --pfile "$PGEN" \
            --extract bed0 "$REGIONS_BED" \
            --keep "$SAMPLES_FILE" \
            --max-alleles 2 \
            --mac 100 \
            --geno 0.05 \
            --hwe 1e-300 \
            --make-pgen \
            --threads "$THREADS" \
            --out "$OUT"
    else
        echo "Skipping chr${CHR}: PGEN file not found at ${PGEN}.pgen"
    fi
done

---
## 02 — Genome-wide merge + LD prune (for REGENIE step 1)

Build the merge list from files produced in step 01, merge into a single PGEN with stable variant IDs, QC-filter, LD-prune, then strip dosages/phase for the REGENIE-step-1 input.


In [ ]:
%%bash
source ./config.sh
source ./common.sh

MERGELIST="${SNP_DIR}/mergeList"
ls "$SNP_DIR"/AoU.chr*.filter.pgen \
    | sed 's/\.pgen$//' \
    | sort -t. -k3,3 -V \
    > "$MERGELIST"

SEED=$(head -n1 "$MERGELIST")
grep -v "^${SEED}$" "$MERGELIST" > "${MERGELIST}.tmp" && mv "${MERGELIST}.tmp" "$MERGELIST"

wc -l "$MERGELIST"  

plink2 --pfile "$SEED" \
       --set-all-var-ids '@:#:$r:$a' \
       --new-id-max-allele-len 20 truncate \
       --pmerge-list "$MERGELIST" \
       --make-pgen \
       --threads "$THREADS" \
       --out "${SNP_DIR}/AoU_88k_TR_CVA_FiltSNPs"

plink2 --pfile "${SNP_DIR}/AoU_88k_TR_CVA_FiltSNPs" \
       --mac 100 --maf 0.01 --geno 0.01 --hwe 1e-300 \
       --write-snplist \
       --threads "$THREADS" \
       --out "${SNP_DIR}/qc_pass"

plink2 --pfile "${SNP_DIR}/AoU_88k_TR_CVA_FiltSNPs" \
       --extract "${SNP_DIR}/qc_pass.snplist" \
       --indep-pairwise 1000 100 0.9 \
       --threads "$THREADS" \
       --out "${SNP_DIR}/pruned_set"

plink2 --pfile "${SNP_DIR}/AoU_88k_TR_CVA_FiltSNPs" \
       --extract "${SNP_DIR}/pruned_set.prune.in" \
       --max-alleles 2 \
       --make-pgen erase-dosage erase-phase \
       --threads "$THREADS" \
       --out "${SNP_DIR}/AoU_88k_TR_CVA_FiltSNPs_LDPrunned_clean"

# REGENIE expects #FID/IID in the psam
fix_psam_header "${SNP_DIR}/AoU_88k_TR_CVA_FiltSNPs_LDPrunned_clean.psam"

---
## 03 — Build per-threshold TR PGEN across EUR/AFR/AMR
Combine the 3 ancestries

PLINK2 → BED (one per population) → PLINK1.9 merge (handles allele flips automatically) → PLINK2 PGEN.

In [ ]:
%%bash
source ./config.sh
source ./common.sh

for THRESH in $(awk 'NR>1 {print $2}' "$TR_TRAIT_PAIRS" | sort -u); do
    echo "=== threshold: $THRESH ==="
    for POP in eur afr amr; do
        plink2 \
            --pfile "${TR_BINARY_INDIR}/${THRESH}/Autosomes_${POP}_EHv5_77k_${THRESH}_BinaryGTForPhewas.BinVar" \
            --extract "${PROJECT_ROOT}/TRList_${THRESH}" \
            --make-bed \
            --threads "$THREADS" \
            --out "${TR_PLINK_DIR}/temp_${POP}_${THRESH}"
    done
    MERGELIST="${TR_PLINK_DIR}/temp_merge_${THRESH}.txt"
    {
        echo "${TR_PLINK_DIR}/temp_afr_${THRESH}"
        echo "${TR_PLINK_DIR}/temp_amr_${THRESH}"
    } > "$MERGELIST"
    plink \
        --bfile "${TR_PLINK_DIR}/temp_eur_${THRESH}" \
        --merge-list "$MERGELIST" \
        --make-bed \
        --out "${TR_PLINK_DIR}/TR_merged_temp_${THRESH}"
    plink2 \
        --bfile "${TR_PLINK_DIR}/TR_merged_temp_${THRESH}" \
        --make-pgen \
        --threads "$THREADS" \
        --out "${TR_PLINK_DIR}/TR_${THRESH}"
    rm -f "${TR_PLINK_DIR}"/temp_*_${THRESH}* "${TR_PLINK_DIR}/TR_merged_temp_${THRESH}"*
done

---
## 04 — Per (TR, PHENO) ±250 kb extraction + drop-SNP list
For each TR/phenotype, extract that locus's SNPs from the filtered chromosome PGEN, set stable variant IDs, then flag SNPs with fewer than two observed genotype classes 

In [ ]:
%%bash
source ./config.sh
source ./common.sh

cd "$PROJECT_ROOT"

OUTDIR="tr_traits/${TYPE}"
mkdir -p "$OUTDIR"

for phenoList in *_Binary_PhenoList; do
    TR=${phenoList%_Binary_PhenoList}
    chr=${TR%%_*}; chr=${chr#chr}
    echo "=== ${TR} (chr${chr}) ==="

    # Per-locus BED with chr prefix stripped (the filter PGEN uses bare chr).
    tr_bed="${TR}.bed"
    awk -v target="$TR" 'BEGIN{OFS="\t"} $4==target { sub(/^chr/,"",$1); print $1,$2,$3 }' \
        ${REGIONS_BED} > "$tr_bed"

    if [[ ! -s "$tr_bed" ]]; then
        echo "WARN: no BED entry for $TR — skipping" >&2
        continue
    fi

    while IFS=$'\t' read -r PHENO _; do
        echo "  -> $PHENO"
        PGEN="SNP/AoU.chr${chr}.filter"
        PGEN2="${OUTDIR}/${TR}_${PHENO}_snps"
        sampleList=$(sample_list_for_plink "$PHENO")

        plink2 --pfile "$PGEN" \
            --keep "$sampleList" \
            --extract range "$tr_bed" \
            --set-all-var-ids '@:#:$r:$a' \
            --new-id-max-allele-len 20 truncate \
            --max-alleles 2 \
            --geno 0.05 --hwe 1e-300 --mac 100 \
            --make-pgen \
            --threads "$THREADS" \
            --out "$PGEN2"

        # Identify SNPs with < 2 observed genotype classes. Hardy file format:
        # the genotype counts are HOM_A1_CT, HET_A1_CT, TWO_AX_CT (plink2 v2+).
        plink2 --pfile "$PGEN2" \
               --keep "$sampleList" \
               --hardy \
               --threads "$THREADS" \
               --out "${OUTDIR}/${TR}_${PHENO}_geno_counts"

        awk '
            NR==1 {
                for (i=1;i<=NF;i++) col[$i]=i;
                # Try the common plink2 column names; pick whichever exists.
                hom=col["HOM_A1_CT"]; het=col["HET_A1_CT"]; two=col["TWO_AX_CT"];
                if (!hom) hom=col["OBS_CT_HOM"];
                if (!het) het=col["OBS_CT_HET"];
                id=col["ID"];
                next
            }
            {
                c = ($(hom)>0) + ($(het)>0) + ($(two)>0)
                if (c<2) print $(id)
            }' "${OUTDIR}/${TR}_${PHENO}_geno_counts.hardy" \
            > "${OUTDIR}/${TR}_${PHENO}_drop_snp.txt"

        rm -f "${OUTDIR}/${TR}_${PHENO}_geno_counts.hardy"

        fix_psam_header "${PGEN2}.psam"
    done < "$phenoList"
done

---
## 05 — REGENIE step 1 (genome-wide LD-pruned model)
One step-1 model per phenotype. Output is reused by every step-2 call

In [ ]:
%%bash
source ./config.sh
source ./common.sh
cd "$PROJECT_ROOT"

plinkFile="SNP/AoU_88k_TR_CVA_FiltSNPs_LDPrunned_clean"

for PHENO in "${STEP1_PHENOS[@]}"; do
    echo "=== step 1: $PHENO ==="
    sampleList=$(sample_list_for_regenie "$PHENO")
    phenoFile=$(phenotype_file "$PHENO")

    "$REGENIE_BIN" --step 1 \
        --pgen "$plinkFile" \
        --keep "$sampleList" \
        --phenoFile "$phenoFile" \
        --phenoCol "$PHENO" \
        --strict \
        --covarFile "$COVAR_FILE" \
        --catCovarList "$CAT_COVARS" \
        --covarColList "$QUANT_COVARS" \
        --out "REGENIE/${TYPE}/${PHENO}_SNPs.step1" \
        --bsize 1000 \
        --bt \
        --loocv \
        --lowmem \
        --lowmem-prefix "REGENIE/${TYPE}/tmp_rg_${PHENO}" \
        --threads "$THREADS" \
        2>&1 | tee "${LOG_DIR}/regenie_step1_${PHENO}.log"
done

---
## 06 — REGENIE step 2 (locus association)
Per (TR, PHENO), associate locus SNPs with the phenotype using the step-1 prediction.

In [ ]:
%%bash
source ./config.sh
source ./common.sh
cd "$PROJECT_ROOT"

for phenoList in *_Binary_PhenoList; do
    TR=${phenoList%_Binary_PhenoList}
    echo "=== $TR ==="

    while IFS=$'\t' read -r PHENO _; do
        echo "  -> $PHENO"
        plinkFile="tr_traits/${TYPE}/${TR}_${PHENO}_snps"
        sampleList=$(sample_list_for_regenie "$PHENO")
        phenoFile=$(phenotype_file "$PHENO")
        excludeList="tr_traits/${TYPE}/${TR}_${PHENO}_drop_snp.txt"
        predList="REGENIE/${TYPE}/${PHENO}_SNPs.step1_pred.list"

        "$REGENIE_BIN" --step 2 \
            --pgen "$plinkFile" \
            --keep "$sampleList" \
            --exclude "$excludeList" \
            --phenoFile "$phenoFile" \
            --phenoCol "$PHENO" \
            --strict \
            --covarFile "$COVAR_FILE" \
            --catCovarList "$CAT_COVARS" \
            --covarColList "$QUANT_COVARS" \
            --out "REGENIE/${TYPE}/${TR}_${PHENO}_SNPs" \
            --bsize 1000 \
            --bt --firth --approx \
            --lowmem \
            --lowmem-prefix "REGENIE/${TYPE}/tmp_rg_${TR}_${PHENO}" \
            --threads "$THREADS" \
            --pred "$predList" \
            --pThresh 0.05 \
            --minMAC 2 \
            2>&1 | tee "${LOG_DIR}/regenie_step2_${TR}_${PHENO}.log"
    done < "$phenoList"
done

---
# Fine mapping (CAVIAR)
## 07 — Z-score table for top-100 SNPs + TR
Pull the top 100 SNPs by `LOG10P` from the per-pair step-2 file, plus the TR row from the upstream combined-cohort REGENIE output, and compute Z = β/SE for each. Column lookups go through `regenie_top_lines` 

In [ ]:
%%bash
source ./config.sh
source ./common.sh
cd "$PROJECT_ROOT"

iterate_pairs | while IFS=$'\t' read -r TR THRESH PHENO; do
    echo "=== $TR $PHENO $THRESH ==="
    INFILE="REGENIE/${TYPE}/${TR}_${PHENO}_SNPs_${PHENO}.regenie"
    TRFILE="${UPSTREAM_REGENIE_DIR}/${THRESH}/${PHENO}/AllAncestries.Mar26.${PHENO}.${THRESH}.${PHENO}.MACounts2.CombinedCohorts.regenie"
    OUT_Z="${CAVIAR_DIR}/${TR}_${PHENO}_${TYPE}_${THRESH}_Top100SNP.z.tsv"
    OUT_IDS="${CAVIAR_DIR}/${TR}_${PHENO}_${TYPE}_${THRESH}_Top100SNP.ids"

    if [[ ! -f "$INFILE" || ! -f "$TRFILE" ]]; then
        echo "  WARN: missing input file(s); skipping" >&2
        continue
    fi

    set +o pipefail
    regenie_top_lines "$INFILE" \
        | sort -k1,1gr \
        | head -n 100 \
        | awk '{ if ($4+0>0) print $2"\t"$3/$4 }' \
        > "$OUT_Z"
    set -o pipefail

    { head -n1 "$TRFILE"; grep -w "$TR" "$TRFILE"; } \
        | regenie_top_lines /dev/stdin \
        | awk -v existing="$OUT_Z" '
            BEGIN { while ((getline l < existing) > 0) { split(l,a,"\t"); seen[a[1]]=1 } }
            $4+0>0 && !($2 in seen) { print $2"\t"$3/$4 }
          ' >> "$OUT_Z" || true

    cut -f1 "$OUT_Z" > "$OUT_IDS"
done

---
## 08 — LD matrix per pair
Build the (top-100 SNPs + TR) merged PGEN, prune perfect-LD SNPs (r²>0.99), then write a square correlation matrix for CAVIAR. Uses the shared `build_tr_snp_merged_pgen` function.

In [ ]:
%%bash
source ./config.sh
source ./common.sh
cd "$PROJECT_ROOT"

iterate_pairs | while IFS=$'\t' read -r TR THRESH PHENO; do
    echo "=== $TR $PHENO $THRESH ==="
    snpList="${CAVIAR_DIR}/${TR}_${PHENO}_${TYPE}_${THRESH}_Top100SNP.ids"
    prefix="${LD_DIR}/${TR}_${PHENO}_${THRESH}_${TYPE}"

    if [[ ! -s "$snpList" ]]; then
        echo "  WARN: snpList missing or empty; skipping" >&2
        continue
    fi

    build_tr_snp_merged_pgen "$TR" "$PHENO" "$THRESH" "$prefix" "$snpList"

    # drop perfect-LD SNPs so CAVIAR's matrix stays well-conditioned
    plink2 --bfile "${prefix}_snps_TR" \
           --extract "$snpList" \
           --indep-pairwise 1000kb 1 0.99 \
           --threads "$THREADS" \
           --out "${prefix}_pruned"

    plink2 --bfile "${prefix}_snps_TR" \
           --extract "${prefix}_pruned.prune.in" \
           --r-unphased square \
           --threads "$THREADS" \
           --out "$prefix"
done

---
## 09 — CAVIAR
Reorder the Z-score file to match the LD-matrix variant order (CAVIAR requires identical ordering), then run CAVIAR. The marker list is built from the filtered TR–trait pairs.

In [ ]:
%%bash
source ./config.sh
source ./common.sh

iterate_pairs | while IFS=$'\t' read -r TR THRESH PHENO; do
    MARKER="${TR}_${PHENO}"
    echo "=== $MARKER ($THRESH) ==="

    zFile="${CAVIAR_DIR}/${MARKER}_${TYPE}_${THRESH}_Top100SNP.z.tsv"
    ldVars="${LD_DIR}/${MARKER}_${THRESH}_${TYPE}.unphased.vcor1.vars"
    outZ="${CAVIAR_DIR}/${MARKER}_${THRESH}_${TYPE}_Top100SNP.ordered.z.tsv"
    ldFile="${LD_DIR}/${MARKER}_${THRESH}_${TYPE}.unphased.vcor1"
    outFile="${CAVIAR_DIR}/${MARKER}_${THRESH}_${TYPE}_Top100SNP.caviar"

    if [[ ! -f "$zFile" || ! -f "$ldVars" || ! -f "$ldFile" ]]; then
        echo "  WARN: missing input(s); skipping" >&2
        continue
    fi

    # CAVIAR requires z and LD matrix to have identical variant ordering
    awk 'BEGIN{FS=OFS="\t"} FNR==NR{z[$1]=$2; next} {print $1, ($1 in z ? z[$1] : "NA")}' \
        "$zFile" "$ldVars" > "$outZ"

    "$CAVIAR_BIN" -l "$ldFile" -z "$outZ" -o "$outFile" -r 0.95 -g 0.01 -c 3 -f1 \
        2>&1 | tee "${LOG_DIR}/caviar_${MARKER}_${THRESH}.log"
done

---
# Conditional analysis
## 10 — Lead SNP per pair + genome-wide exclude list
Pull the top-1 SNP (by `LOG10P`, NA excluded) from each per-pair step-2 file, then build a deduplicated list to exclude from the conditional REGENIE step 1.

In [ ]:
%%bash
source ./config.sh
source ./common.sh

iterate_pairs | while IFS=$'\t' read -r TR THRESH PHENO; do
    INFILE="${REGENIE_DIR}/${TYPE}/${TR}_${PHENO}_SNPs_${PHENO}.regenie"
    OUTFILE="${COND_DIR}/${TR}_${PHENO}_${TYPE}_${THRESH}_LeadSNP.tsv"

    if [[ ! -f "$INFILE" ]]; then
        echo "WARN: $INFILE missing; skipping" >&2
        continue
    fi

    set +o pipefail
    regenie_top_lines "$INFILE" \
        | sort -k1,1gr \
        | head -n1 \
        | awk '{print $2}' > "$OUTFILE"
    set -o pipefail
done

# Deduplicated exclude list for step 1.
cat "${COND_DIR}"/*_LeadSNP.tsv | sort -u \
    > "${COND_DIR}/LeadSNPsToExcludeFromRegenieStep1.tsv"

pvarIds=$(mktemp)
awk '!/^#/ {print $3}' "${SNP_DIR}/AoU_88k_TR_CVA_FiltSNPs_LDPrunned_clean.pvar" | sort -u > "$pvarIds"
nExclude=$(wc -l < "${COND_DIR}/LeadSNPsToExcludeFromRegenieStep1.tsv")
nMatched=$(comm -12 "$pvarIds" <(sort -u "${COND_DIR}/LeadSNPsToExcludeFromRegenieStep1.tsv") | wc -l)
echo "Lead-SNP exclude list: $nExclude entries, $nMatched present in LD-pruned PGEN"
rm -f "$pvarIds"

---
## 11 — Build per-pair merged TR + locus PGEN for conditional step 2
Same merging logic as step 08, but no SNP subset — keep all locus SNPs. Uses the shared `build_tr_snp_merged_pgen` function with no extract list.

In [ ]:
%%bash
source ./config.sh
source ./common.sh

iterate_pairs | while IFS=$'\t' read -r TR THRESH PHENO; do
    echo "=== $TR $PHENO $THRESH ==="
    prefix="${COND_DIR}/${TR}_${PHENO}_${THRESH}_${TYPE}"
    build_tr_snp_merged_pgen "$TR" "$PHENO" "$THRESH" "$prefix"

    plink2 --bfile "${prefix}_snps_TR" \
           --make-pgen \
           --threads "$THREADS" \
           --out "${COND_DIR}/${TR}_${PHENO}_${TYPE}_${THRESH}_snps_TR"
done

---
## 12 — Conditional REGENIE step 1 (lead SNPs excluded)
Re-run step 1 with the lead SNPs from every TR-trait pair removed. One model per phenotype.

In [ ]:
%%bash
source ./config.sh
source ./common.sh

PGEN="${SNP_DIR}/AoU_88k_TR_CVA_FiltSNPs_LDPrunned_clean"
excludeLeadSNP="${COND_DIR}/LeadSNPsToExcludeFromRegenieStep1.tsv"

for PHENO in "${STEP1_PHENOS[@]}"; do
    echo "=== conditional step 1: $PHENO ==="
    sampleList=$(sample_list_for_regenie "$PHENO")
    phenoFile=$(phenotype_file "$PHENO")

    "$REGENIE_BIN" --step 1 \
        --pgen "$PGEN" \
        --keep "$sampleList" \
        --exclude "$excludeLeadSNP" \
        --phenoFile "$phenoFile" \
        --phenoCol "$PHENO" \
        --strict \
        --covarFile "$COVAR_FILE" \
        --catCovarList "$CAT_COVARS" \
        --covarColList "$QUANT_COVARS" \
        --out "${COND_STEP1_DIR}/${PHENO}_${TYPE}_SNPs.step1" \
        --bt --bsize 1000 \
        --firth --approx \
        --loocv \
        --lowmem \
        --lowmem-prefix "${COND_STEP1_DIR}/tmp_rg_${PHENO}" \
        --threads "$THREADS" \
        2>&1 | tee "${LOG_DIR}/regenie_cond_step1_${PHENO}.log"
done

---
## 13 — Conditional REGENIE step 2: condition on lead SNP
Per-pair step 2 conditioning on the lead SNP found in step 10. Uses `run_regenie_step2_conditional` for the shared call.

In [ ]:
%%bash
source ./config.sh
source ./common.sh

iterate_pairs | while IFS=$'\t' read -r TR THRESH PHENO; do
    echo "=== condition-on-lead-SNP: $TR $PHENO $THRESH ==="
    condition="${COND_DIR}/${TR}_${PHENO}_${TYPE}_${THRESH}_LeadSNP.tsv"
    if [[ ! -s "$condition" ]]; then
        echo "  WARN: empty lead-SNP file; skipping" >&2
        continue
    fi
    # minMAC=2 matches the unconditioned step 2
    run_regenie_step2_conditional "$TR" "$PHENO" "$THRESH" \
        "$condition" "snps_Conditional" 2 \
        2>&1 | tee "${LOG_DIR}/regenie_cond_step2_snp_${TR}_${PHENO}.log"
done

---
## 14 — Conditional REGENIE step 2: condition on TR
Mirror of step 13 but conditioning on the TR rather than the lead SNP. 

In [ ]:
%%bash
source ./config.sh
source ./common.sh

unique_trs | while IFS= read -r TR; do
    printf '%s\n' "$TR" > "${COND_DIR}/${TR}_LeadTR.tsv"
done

iterate_pairs | while IFS=$'\t' read -r TR THRESH PHENO; do
    echo "=== condition-on-TR: $TR $PHENO $THRESH ==="
    condition="${COND_DIR}/${TR}_LeadTR.tsv"
    run_regenie_step2_conditional "$TR" "$PHENO" "$THRESH" \
        "$condition" "TR_Conditional" 5 \
        2>&1 | tee "${LOG_DIR}/regenie_cond_step2_tr_${TR}_${PHENO}.log"
done